<a href="https://www.kaggle.com/code/adegbaju/global-mmlu-lite-evaluation?scriptVersionId=302257438" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

#  Global MMLU Lite Evaluation 

In [1]:
!pip install pandas pyarrow transformers torch tqdm

# Setup logging

In [2]:
import os
import pandas as pd
import logging
import torch
from tqdm.notebook import tqdm
from typing import List, Dict, Any, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM
print(torch.cuda.is_available())

True


# Dataset Manager

In [3]:


# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class DatasetManager:
    def __init__(self, data_dir: str = "/kaggle/input/datasets/organizations/cohere-labs/global-mmlu-lite"):
        self.data_dir = data_dir
        self.data_cache = {}
        self._available_languages = None  # lazy-loaded

    def _get_language_from_filename(self, filename: str) -> Optional[str]:
        """Extract language code from filename like 'Global-MMLU-Lite-ar-test.parquet'."""
        if not filename.startswith("Global-MMLU-Lite-") or not filename.endswith(".parquet"):
            return None
        # Remove prefix and suffix
        core = filename.replace("Global-MMLU-Lite-", "").replace(".parquet", "")
        # Split by '-', the language code is the first part (e.g., 'ar-test' -> 'ar')
        parts = core.split('-')
        return parts[0] if parts else None

    def get_available_languages(self) -> List[str]:
        """Return sorted list of language codes for which files exist."""
        if self._available_languages is None:
            languages = set()
            for fname in os.listdir(self.data_dir):
                lang = self._get_language_from_filename(fname)
                if lang:
                    languages.add(lang)
            self._available_languages = sorted(languages)
        return self._available_languages.copy()

    def load_dataset(self, language: str, split: str = "test") -> pd.DataFrame:
        """
        Load dataset for a specific language and split.
        Supported splits: 'dev', 'test'.
        """
        if split not in ["dev", "test"]:
            raise ValueError(f"Split must be 'dev' or 'test', got '{split}'")

        cache_key = f"{language}_{split}"
        if cache_key in self.data_cache:
            return self.data_cache[cache_key].copy()

        filename = f"Global-MMLU-Lite-{language}-{split}.parquet"
        filepath = os.path.join(self.data_dir, filename)

        if not os.path.exists(filepath):
            # Fallback: try to discover by scanning (in case of naming variations)
            available = self.get_available_languages()
            if language not in available:
                raise FileNotFoundError(
                    f"Language '{language}' not found. Available languages: {available}"
                )
            # If language is available but file not found, something else is wrong
            raise FileNotFoundError(f"Dataset file not found: {filepath}")

        logger.info(f"Loading {split} split for language '{language}'")
        df = pd.read_parquet(filepath)
        df['language'] = language
        df['split'] = split
        df['question_id'] = range(len(df))  # simple index
        self.data_cache[cache_key] = df
        logger.info(f"Loaded {len(df)} examples")
        return df.copy()

    def load_all_splits(self, languages: Optional[List[str]] = None) -> Dict[str, pd.DataFrame]:
        """
        Load both dev and test for specified languages (or all available).
        Returns a dict with keys like 'en_test', 'en_dev', etc.
        """
        if languages is None:
            languages = self.get_available_languages()
        result = {}
        for lang in languages:
            for split in ['dev', 'test']:
                try:
                    df = self.load_dataset(lang, split)
                    result[f"{lang}_{split}"] = df
                except FileNotFoundError:
                    logger.warning(f"Could not load {lang} {split}")
        return result

    def get_dataset_info(self, language: str) -> Dict[str, Any]:
        """Return summary statistics for a language."""
        try:
            dev_df = self.load_dataset(language, "dev")
            test_df = self.load_dataset(language, "test")
            combined = pd.concat([dev_df, test_df], ignore_index=True)

            info = {
                "language": language,
                "dev_samples": len(dev_df),
                "test_samples": len(test_df),
                "total_samples": len(combined),
                "subjects": sorted(combined['subject'].unique().tolist()),
                "cultural_sensitivity_distribution": {
                    "CS": combined[combined['cultural_sensitivity_label'] == 'CS'].shape[0],
                    "CA": combined[combined['cultural_sensitivity_label'] == 'CA'].shape[0],
                },
                "subject_category_distribution": combined['subject_category'].value_counts().to_dict()
            }
            return info
        except Exception as e:
            logger.error(f"Failed to get info for {language}: {e}")
            return {"language": language, "error": str(e)}

In [4]:
# Initialize the manager
dm = DatasetManager()

# See which languages are available
print("Available languages:", dm.get_available_languages())

# Load test data for English and Arabic
en_test = dm.load_dataset("en", "test")
ar_test = dm.load_dataset("ar", "test")

print(f"English test samples: {len(en_test)}")
print(f"Arabic test samples: {len(ar_test)}")

# Get info for one language
info = dm.get_dataset_info("fr")
print(f"French dataset info: {info}")

# Load all splits for all languages (may take a while)
all_data = dm.load_all_splits()
print(f"Loaded {len(all_data)} dataframes")

INFO:__main__:Loading test split for language 'en'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading test split for language 'ar'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading dev split for language 'fr'


Available languages: ['ar', 'bn', 'cy', 'de', 'en', 'es', 'fr', 'hi', 'id', 'it', 'ja', 'ko', 'my', 'pt', 'sq', 'sw', 'yo', 'zh']
English test samples: 400
Arabic test samples: 400


INFO:__main__:Loaded 215 examples
INFO:__main__:Loading test split for language 'fr'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading dev split for language 'ar'


French dataset info: {'language': 'fr', 'dev_samples': 215, 'test_samples': 400, 'total_samples': 615, 'subjects': ['astronomy', 'business_ethics', 'college_biology', 'college_chemistry', 'college_mathematics', 'college_medicine', 'electrical_engineering', 'elementary_mathematics', 'global_facts', 'high_school_chemistry', 'high_school_computer_science', 'high_school_european_history', 'high_school_geography', 'high_school_government_and_politics', 'high_school_macroeconomics', 'high_school_mathematics', 'high_school_microeconomics', 'high_school_physics', 'high_school_psychology', 'high_school_statistics', 'high_school_us_history', 'high_school_world_history', 'human_aging', 'human_sexuality', 'international_law', 'jurisprudence', 'logical_fallacies', 'management', 'marketing', 'miscellaneous', 'moral_disputes', 'nutrition', 'philosophy', 'prehistory', 'professional_accounting', 'professional_law', 'professional_medicine', 'professional_psychology', 'public_relations', 'security_studie

INFO:__main__:Loaded 215 examples
INFO:__main__:Loading dev split for language 'bn'
INFO:__main__:Loaded 215 examples
INFO:__main__:Loading test split for language 'bn'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading test split for language 'cy'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading dev split for language 'de'
INFO:__main__:Loaded 215 examples
INFO:__main__:Loading test split for language 'de'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading dev split for language 'en'
INFO:__main__:Loaded 215 examples
INFO:__main__:Loading dev split for language 'es'
INFO:__main__:Loaded 215 examples
INFO:__main__:Loading test split for language 'es'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading dev split for language 'hi'
INFO:__main__:Loaded 215 examples
INFO:__main__:Loading test split for language 'hi'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading dev split for language 'id'
INFO:__main__:Loaded 215 examples
INFO:__main__:Loading test split for 

Loaded 35 dataframes


# Evaluation functions

In [5]:
def format_prompt(row):
    """Format a multiple-choice question."""
    prompt = f"Question: {row['question']}\n"
    prompt += f"A. {row['option_a']}\n"
    prompt += f"B. {row['option_b']}\n"
    prompt += f"C. {row['option_c']}\n"
    prompt += f"D. {row['option_d']}\n"
    prompt += "Answer:"
    return prompt

def predict_answer(prompt, tokenizer, model, max_new_tokens=10):
    """Generate answer and extract letter (A, B, C, D)."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Find the part after "Answer:"
    if "Answer:" in generated:
        answer_part = generated.split("Answer:")[-1].strip()
    else:
        answer_part = generated[len(prompt):].strip()
    for token in answer_part:
        if token.upper() in "ABCD":
            return token.upper()
    return None  # fallback

# Main script

In [6]:


def main():
    # - Configuration -
   
    DATA_DIR = "/kaggle/input/datasets/organizations/cohere-labs/global-mmlu-lite"
    MODEL_NAME = "microsoft/phi-3-mini-4k-instruct"         # 13B model – requires ~26GB GPU
   
    SAMPLES_PER_LANGUAGE = 10                    # Use None to evaluate all samples
    # -----------------------------------------------------

    # Initialize dataset manager
    dm = DatasetManager(DATA_DIR)
    available_langs = dm.get_available_languages()
    print(f"Available languages: {available_langs}")

    # Build test set: take up to SAMPLES_PER_LANGUAGE from each language
    test_dfs = []
    for lang in available_langs:
        try:
            df = dm.load_dataset(lang, "test")
            if SAMPLES_PER_LANGUAGE:
                df = df.head(SAMPLES_PER_LANGUAGE)
            test_dfs.append(df)
        except FileNotFoundError:
            print(f"Test split not found for {lang}, skipping.")
    if not test_dfs:
        print("No test data loaded. Exiting.")
        return
    eval_df = pd.concat(test_dfs, ignore_index=True)
    print(f"Evaluating on {len(eval_df)} samples.")

    # Load model and tokenizer
    print(f"Loading model {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    model.eval()
    print("Model loaded.")

    # Run inference
    results = []
    for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating"):
        prompt = format_prompt(row)
        pred = predict_answer(prompt, tokenizer, model)
        results.append({
            "language": row["language"],
            "subject_category": row["subject_category"],
            "cultural_sensitivity_label": row["cultural_sensitivity_label"],
            "true_answer": row["answer"],
            "predicted": pred,
            "correct": pred == row["answer"]
        })

    results_df = pd.DataFrame(results)

    # Compute and display metrics
    overall_acc = results_df["correct"].mean() * 100
    print(f"\nOverall accuracy: {overall_acc:.2f}%")

    print("\nAccuracy by language:")
    lang_acc = results_df.groupby("language")["correct"].mean().mul(100).round(2)
    print(lang_acc.sort_values(ascending=False))

    print("\nAccuracy by subject category:")
    cat_acc = results_df.groupby("subject_category")["correct"].mean().mul(100).round(2)
    print(cat_acc)

    print("\nAccuracy by cultural sensitivity label:")
    cs_acc = results_df.groupby("cultural_sensitivity_label")["correct"].mean().mul(100).round(2)
    print(cs_acc)

    # Optional: save detailed results
    results_df.to_csv("gmmlu_lite_results.csv", index=False)
    print("\nDetailed results saved to 'gmmlu_lite_results.csv'.")

if __name__ == "__main__":
    main()

INFO:__main__:Loading test split for language 'ar'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading test split for language 'bn'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading test split for language 'cy'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading test split for language 'de'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading test split for language 'en'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading test split for language 'es'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading test split for language 'fr'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading test split for language 'hi'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading test split for language 'id'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading test split for language 'it'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading test split for language 'ja'
INFO:__main__:Loaded 400 examples
INFO:__main__:Loading test split for language 'ko'
INFO:__main__:

Available languages: ['ar', 'bn', 'cy', 'de', 'en', 'es', 'fr', 'hi', 'id', 'it', 'ja', 'ko', 'my', 'pt', 'sq', 'sw', 'yo', 'zh']


INFO:__main__:Loaded 400 examples


Evaluating on 180 samples.
Loading model microsoft/phi-3-mini-4k-instruct...


INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/phi-3-mini-4k-instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3-mini-4k-instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/Phi-3-mini-4k-instruct/f39ac1d28e925b323eae81227eaba4464caced4e/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/Phi-3-mini-4k-instruct/f39ac1d28e925b323eae81227eaba4464caced4e/config.json "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/phi-3-mini-4k-instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3-mini-4k-instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/Phi-3-mini-4k-instruct/f39ac1d28e925b323eae81227eaba4464caced4e/generation_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/Phi-3-mini-4k-instruct/f39ac1d28e925b323eae81227eaba4464caced4e/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded.


Evaluating:   0%|          | 0/180 [00:00<?, ?it/s]


Overall accuracy: 38.89%

Accuracy by language:
language
ko    70.0
en    70.0
zh    60.0
bn    60.0
my    50.0
sw    40.0
ja    40.0
de    40.0
id    40.0
hi    40.0
ar    30.0
sq    30.0
es    30.0
fr    30.0
it    20.0
cy    20.0
pt    20.0
yo    10.0
Name: correct, dtype: float64

Accuracy by subject category:
subject_category
Business    43.38
STEM        17.65
Name: correct, dtype: float64

Accuracy by cultural sensitivity label:
cultural_sensitivity_label
CA    40.74
CS    38.10
Name: correct, dtype: float64

Detailed results saved to 'gmmlu_lite_results.csv'.


# Interpretation of Results

### **Overall accuracy**: 38.89%
This is well below the ~70‑80% you would expect from a strong multilingual model like Aya‑101. It makes sense because Phi‑3 Mini is primarily an English‑centric model – it was trained mostly on English data. It can handle some other languages to a limited extent (especially if they share the Latin script or were seen during training), but it is not designed for multilingual evaluation.

**Per‑language breakdown**
English (70%) and Korean (70%) perform best.

English is the model's main language.

Korean's strong performance may be due to the tokenizer handling Hangul reasonably well and some Korean data in training.

Chinese (60%), Bengali (60%), Burmese (50%) show moderate results – perhaps because they use scripts that are less common in the training corpus.

Arabic, Albanian, Spanish, French (30%) are much lower.

Yoruba (10%) is near random (25% chance), indicating the model has almost no knowledge of Yoruba.

**Per subject category**
Business (43.38%) is higher than STEM (17.65%) – this may reflect that business questions are more language‑agnostic or that the model has seen more business text.

**Cultural sensitivity**
CA (40.74%) vs CS (38.10%) – a small gap, but not statistically significant given the small sample (10 per language). The full set might show a clearer difference.